In [2]:
%pip install tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 MB 12.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 13.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 6.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 12.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [tensorflow]8 [tensorflow]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout

# SECTION 1: Preprecessing


In [4]:
import os

In [5]:
EPOCH_LENGTH = 480

# ระบุ path ของโฟลเดอร์ที่เก็บไฟล์ CSV
folder_path = 'train/train'  # เปลี่ยนเป็น path ของคุณ
csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

# ---------------------------
# Load and Combine Data from Multiple CSV Files
# ---------------------------
dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

# รวม DataFrame ทั้งหมดเข้าด้วยกัน
data_all = pd.concat(dataframes, ignore_index=True)

# ---------------------------
# Preprocess the Data
# ---------------------------
# สมมุติว่าคอลัมน์ "Sleep_Stage" เป็น label ส่วนที่เหลือคือข้อมูลเซ็นเซอร์
sensor_cols = [col for col in data_all.columns if col != 'Sleep_Stage']

# คำนวณจำนวน epoch ที่ครบถ้วน
n_epochs = len(data_all) // EPOCH_LENGTH
print('epoch :', n_epochs)
data_all = data_all.iloc[:n_epochs * EPOCH_LENGTH]

# แปลงข้อมูลเซ็นเซอร์เป็น array และจัดรูปเป็น (n_epochs, 480, n_features)
data_array = data_all[sensor_cols].values
X = data_array.reshape(n_epochs, EPOCH_LENGTH, len(sensor_cols))

# สำหรับ label สมมุติว่า label ในแต่ละ epoch คงที่ จึงดึง label จาก row แรกของแต่ละ epoch
labels = data_all['Sleep_Stage'].values
y = labels[::EPOCH_LENGTH]

# แปลง label เป็นตัวเลขและ one-hot encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)

# แบ่งข้อมูลเป็น train/test set
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=1)

epoch : 66745


## Model

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, BatchNormalization, Bidirectional

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dropout, Dense, BatchNormalization

# ตรวจสอบว่ามี GPU อยู่หรือไม่
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Using GPU for training")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected. Running on CPU")
with tf.device('/GPU:0'):
    model = Sequential()
    # เลเยอร์แรกใช้ Bidirectional LSTM
    model.add(Bidirectional(LSTM(128, return_sequences=True), input_shape=(EPOCH_LENGTH, len(sensor_cols))))
    model.add(Dropout(0.3))
    # เพิ่มเลเยอร์ Bidirectional LSTM อีกชั้น
    model.add(Bidirectional(LSTM(128, return_sequences=True)))
    model.add(Dropout(0.3))
    # เลเยอร์สุดท้ายของ LSTM โดยไม่ต้อง return sequences
    model.add(Bidirectional(LSTM(128)))
    model.add(Dropout(0.3))
    # เพิ่ม Dense layers พร้อม BatchNormalization
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(y_cat.shape[1], activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.summary()

No GPU detected. Running on CPU


/opt/anaconda3/envs/nlp-env/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 480, 256)       │       140,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 480, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 480, 256)       │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 480, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 996,869 (3.80 MB)

 Trainable params: 996,357 (3.80 MB)

 Non-trainable params: 512 (2.00 KB)

## Train model

In [ ]:
history = model.fit(X_train, y_train, 
                    epochs=100, 
                    batch_size=32, 
                    validation_split=0.1)

Epoch 1/100
1502/1502 ━━━━━━━━━━━━━━━━━━━━ 2912s 2s/step - accuracy: 0.4988 - loss: 1.3330 - val_accuracy: 0.5386 - val_loss: 1.3722
Epoch 2/100
 185/1502 ━━━━━━━━━━━━━━━━━━━━ 59:08 3s/step - accuracy: 0.5444 - loss: 1.2377

## Evaluation Model

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

# Evaluation

In [ ]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

f1_each = f1_score(y_true, y_pred, average=None)
print("F1 score for each class:", f1_each)

report = classification_report(y_true, y_pred, target_names=le.classes_)
print(report)

### Test

In [ ]:
test = pd.read_csv('test_segment/test_segment/test001/test001_00000.csv')
test.shape

# Deploy

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# กำหนดความยาวของ epoch (ในที่นี้ 480 rows = 30 วินาทีที่ 16 Hz)
EPOCH_LENGTH = 480

# รายการสำหรับเก็บผลลัพธ์
results = []

# โฟลเดอร์ที่เก็บชุดทดสอบ
test_segment_dir = '/kaggle/input/io-t-sleep-stage-classification-version-2/test_segment/test_segment'

# ค้นหาโฟลเดอร์ test001 ถึง test010
test_folders = sorted(glob.glob(os.path.join(test_segment_dir, 'test*')))
for folder in test_folders:
    # ค้นหาไฟล์ CSV ในแต่ละโฟลเดอร์
    csv_files = sorted(glob.glob(os.path.join(folder, '*.csv')))
    for csv_file in csv_files:
        print('file :', csv_file)
        # โหลดข้อมูลจากไฟล์ CSV
        df = pd.read_csv(csv_file)
        
        # ตรวจสอบว่ามีจำนวนแถวไม่น้อยกว่า EPOCH_LENGTH
        if len(df) < EPOCH_LENGTH:
            print(f"Warning: {csv_file} มีจำนวนแถวไม่ถึง {EPOCH_LENGTH}. ข้ามไฟล์นี้")
            continue
        
        # ใช้เฉพาะข้อมูล EPOCH_LENGTH แถวแรก
        df = df.iloc[:EPOCH_LENGTH]
        
        # แปลงข้อมูลเป็น numpy array และ reshape ให้มีรูปแบบ (1, EPOCH_LENGTH, num_features)
        data = df.values
        sample = np.expand_dims(data, axis=0)  # เพิ่ม dimension สำหรับ batch
        
        # ทำนาย label โดยใช้โมเดล
        pred_prob = model.predict(sample)
        print('labels :', pred_prob)
        pred_index = np.argmax(pred_prob, axis=1)[0]


                # ถ้ามี LabelEncoder (le) จากขั้นตอนเทรน เราสามารถแปลง index เป็น label ได้
        pred_label = le.inverse_transform([pred_index])[0]
        
        # เก็บผลลัพธ์: ชื่อไฟล์และ label ที่ทำนายได้
        results.append({
            'id': os.path.basename(csv_file),
            'labels': pred_label
        })

# สร้าง DataFrame จากผลลัพธ์และบันทึกเป็นไฟล์ CSV
results_df = pd.DataFrame(results)
results_df['id'] = results_df['id'].str.replace('.csv', '', regex=False)
results_df.to_csv('test_predictions.csv', index=False)

print("เสร็จสิ้นการทำนายและบันทึกผลลัพธ์ในไฟล์ 'test_predictions.csv'")